# Data Operations Pipeline
This notebook demonstrates how to implement a chain of operations on DataFrames representing EU countries over years.
We use `pandas` DataFrames because they inherently handle row (countries) and column (years) indices perfectly, and support underlying numpy matrix operations natively.
We also introduce an `Operation` class that overloads the `>>` operator to compose functions, acting like a monoid for function composition (a functional delegate pattern).

In [ ]:
import pandas as pd
import numpy as np

countries = ['DE', 'FR', 'IT', 'ES', 'PL']

# Create sample data tables T_i
np.random.seed(42)
years_T1 = list(range(2005, 2025))
T_1 = pd.DataFrame(np.random.rand(len(countries), len(years_T1)), index=countries, columns=years_T1)

years_T2 = [2018, 2019, 2020, 2021, 2022]
T_2 = pd.DataFrame(np.random.rand(len(countries), len(years_T2)), index=countries, columns=years_T2)

cols_T3 = ['y1', 'y2', 'y3', 'y4', 'y5']
T_3 = pd.DataFrame(np.random.rand(len(countries), len(cols_T3)), index=countries, columns=cols_T3)

display(T_1.head(), T_2.head(), T_3.head())

## Operation Composer (Monoid Pattern)
We define an `Operation` class that wraps a function. The `>>` operator is overloaded to compose two operations together. This creates a highly readable, declarative pipeline syntax.

In [ ]:
class Operation:
    def __init__(self, func):
        self.func = func
        
    def __call__(self, x):
        return self.func(x)
        
    def __rshift__(self, other):
        # Monoid-like composition: op1 >> op2
        return Operation(lambda x: other(self.func(x)))

# Factory functions to generate operations with parameters
def select_years(start, end):
    return Operation(lambda df: df.loc[:, start:end])

def select_specific_years(years):
    return Operation(lambda df: df[years])

def select_column(col):
    return Operation(lambda df: df[col])

def weighted_average(weights):
    # Matrix multiplication handles the weighted sum across columns naturally
    # df shape: (N_countries, N_years), weights shape: (N_years,)
    # Pandas @ operator delegates to numpy matrix multiplication
    return Operation(lambda df: df @ weights)

# Parameter-less operations
arithmetic_average = Operation(lambda df: df.mean(axis=1))
apply_sqrt = Operation(lambda s: np.sqrt(s))
apply_log = Operation(lambda s: np.log(s))

## A_1: Weighted average for years 2010-2019

In [ ]:
# Years 2010 to 2019 inclusive -> 10 years
weights_A1 = np.ones(10) / 10  # Example equal weights, could be any vector

# Build the pipeline
pipeline_1 = select_years(2010, 2019) >> weighted_average(weights_A1)

# Execute 
A_1 = pipeline_1(T_1)
display(A_1)

## A_2: Arithmetic average for specific years, then sqrt

In [ ]:
target_years = [2018, 2020, 2022]

pipeline_2 = select_specific_years(target_years) >> arithmetic_average >> apply_sqrt

A_2 = pipeline_2(T_2)
display(A_2)

## A_3: Log of column 'y4'

In [ ]:
pipeline_3 = select_column('y4') >> apply_log

A_3 = pipeline_3(T_3)
display(A_3)

## Task 2: Conditional Multiple Array Construction
Given multiple arrays $A_1, A_2, \dots$, we have corresponding weights $W_1, W_2$. 
A 1D Boolean mask $M_i$ (indexed by countries) decides which weight array to apply to the row.
If $M_i$ is True, we use $W_1$; if False, we use $W_2$.

In [ ]:
np.random.seed(42)
A_1 = pd.DataFrame(np.random.rand(5, 3), index=countries, columns=['y1', 'y2', 'y3'])
A_2 = pd.DataFrame(np.random.rand(5, 3), index=countries, columns=['y1', 'y2', 'y3'])

# W_1 and W_2 apply based on the boolean mask evaluation per country.
# Assuming lengths equal to number of A_i (e.g., 2 in this case)
W_1 = np.array([0.7, 0.3])
W_2 = np.array([0.4, 0.6])

# 1D Mask with index aligned to countries
M_1 = pd.Series([True, False, True, False, True], index=countries, name='mask')

def conditional_weighting(mask, weight_true, weight_false):
    # Returns an operation that conditionally weights a list of DataFrames based on the 1D mask.
    # Note: the input is expected to be a tuple/list of DataFrames (e.g., (A_1, A_2)).
    return Operation(lambda inputs: (
        # For each A_i, we multiply its rows where mask is True by the corresponding element in weight_true,
        # and where mask is False by the element in weight_false. 
        # The `mul` operation with `axis=0` broadcasts perfectly across the columns.
        sum([
            # `np.where(mask, w_t, w_f)` constructs the correctly shaped country-level weights.
            A_i.mul(np.where(mask, weight_true[i], weight_false[i]), axis=0)
            for i, A_i in enumerate(inputs)
        ])
    ))

# Constructing the pipeline for an aggregate A_x using the conditional weights:
apply_weights = conditional_weighting(M_1, W_1, W_2)

# Execute pipeline on the list of arrays
A_x = apply_weights((A_1, A_2))

display("Mask (", M_1, ")")
display("Weighted Aggregate A_x:", A_x)